In [1]:
%load_ext autoreload
%autoreload 2

# Other useful imports
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.signal as signal
from sklearn.preprocessing import scale
from sklearn.decomposition import PCA, FastICA
import time
from scipy.signal import welch
from IPython.display import clear_output
import scipy.stats as stats

# Spyeeg import
from spyeeg.generate import simulate_channels, simulate_multisensory_channels, simulate_continuous_stimuli, simulate_PAC
import spyeeg.models as models


In [4]:
mpl.rcParams.update(mpl.rcParamsDefault)

%matplotlib inline
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=["b", "r", "g"])
mpl.rcParams['axes.grid'] = True
mpl.rcParams['axes.linewidth'] = 0.4
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['font.size'] = 3.5
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['grid.alpha'] = 0.4
mpl.rcParams['grid.linewidth'] = 0.2
mpl.rcParams['lines.linewidth'] = 0.6


In [291]:
# Generate the Data without noise
fs = 50
n_feat = 1
n_channels = 4
T = 6000
time_array, X, Y, events, impulse_responses = simulate_channels(n_feat = n_feat, n_channels = n_channels, fs = fs, T = T,
                                                              stim_type = 'continuous',
                                                              snr_db = -20, beta_noise = 0.4, 
                                                              impulse_freqs = [0.1,10], decreasing_rates = [1,20], 
                                                              delays = [0.00,0.2], 
                                                              filter_impulse = False, filter_val = [0.01,20],
                                                              random_seed = 0)


_, X_noise, Y_noise, _, _ = simulate_channels(n_feat = n_feat, n_channels = 30, fs = fs, T = T,
                                                              stim_type = 'continuous',
                                                              snr_db = -50, beta_noise = 0.4, 
                                                              impulse_freqs = [0.1,10], decreasing_rates = [1,20], 
                                                              delays = [0.00,0.2], 
                                                              filter_impulse = False, filter_val = [0.01,20],
                                                              random_seed = 10)


In [292]:
Yadd = np.zeros((Y.shape[0], 10))

for i in range(Yadd.shape[1]):
    for j in range(Y.shape[1]):
        Yadd[:,i] += Y[:,j] * np.random.random() 
    #Yadd[:,i] += Y_noise[:,i]


Y = scale(Yadd)

In [293]:
tmin, tmax = -.5, 1
alphas = [0] + list(np.logspace(-3,3,5))
n_folds = 5
metrics = 'corr'

In [294]:
trf = models.TRF.TRFEstimator(tmin=tmin, tmax=tmax, srate = fs, alpha = alphas, fit_domain = 'time')

# Now, we can fit the data using cross-fold validation
scores_spyeeg = trf.xval_eval(X, Y, n_splits = n_folds, scoring = metrics,verbose = False)

scores_spyeeg = scores_spyeeg
reg = scores_spyeeg.mean(0).mean(0).argmax()

# corr in samples
trf = models.TRF.TRFEstimator(tmin=tmin, tmax=tmax, srate = fs, alpha = alphas, fit_domain = 'time')
trf.fit(X,Y)
Yhat_trf = trf.predict(X)

scores_spyeeg = []
for i in range(Yhat_trf.shape[1]):
    scores_chan = []
    for j in range(Yhat_trf.shape[-1]):
        scores_chan.append(np.corrcoef(Y[:,i], Yhat_trf[:,i,j])[0,1])
    scores_spyeeg.append(scores_chan)

In [299]:
import numpy as np
from spyeeg.models._methods import fit_iRRR_fista

In [296]:
X2,Y2 = trf.get_XY(X,Y)

scores_iRRR = []
for lam in [0] + list(np.logspace(2,3,10)):
    scores_chan = []
    B, info = fit_iRRR_fista(X2, Y2, lam=lam, max_iter=1000, tol=1e-7, verbose=True)
    Yhat = (X2 - info["X_mean"]) @ B + info["Y_mean"]
    for i in range(Yhat.shape[1]):
        scores_chan.append(np.corrcoef(Y2[:,i], Yhat[:,i])[0,1])
    scores_iRRR.append(scores_chan)



iter    1  obj=1.478367e+06  rank= 10
iter   25  obj=1.469628e+06  rank= 10  rel_obj=2.18e-05  rel_step=1.31e-02
iter   50  obj=1.469263e+06  rank= 10  rel_obj=4.55e-06  rel_step=5.23e-03
iter   75  obj=1.469162e+06  rank= 10  rel_obj=1.70e-06  rel_step=3.02e-03
iter  100  obj=1.469118e+06  rank= 10  rel_obj=8.88e-07  rel_step=2.11e-03
iter  125  obj=1.469092e+06  rank= 10  rel_obj=5.56e-07  rel_step=1.63e-03
iter  150  obj=1.469075e+06  rank= 10  rel_obj=3.85e-07  rel_step=1.34e-03
iter  175  obj=1.469063e+06  rank= 10  rel_obj=2.82e-07  rel_step=1.13e-03
iter  200  obj=1.469054e+06  rank= 10  rel_obj=2.13e-07  rel_step=9.73e-04
iter  225  obj=1.469047e+06  rank= 10  rel_obj=1.65e-07  rel_step=8.49e-04
iter  250  obj=1.469042e+06  rank= 10  rel_obj=1.30e-07  rel_step=7.48e-04
iter  275  obj=1.469038e+06  rank= 10  rel_obj=1.04e-07  rel_step=6.63e-04
iter  300  obj=1.469034e+06  rank= 10  rel_obj=8.35e-08  rel_step=5.91e-04
iter  325  obj=1.469032e+06  rank= 10  rel_obj=6.76e-08  rel_s

In [297]:
for i in range(np.asarray(scores_iRRR).shape[1]):
    print(np.asarray(scores_iRRR)[-1,i] - np.max(np.asarray(scores_spyeeg)[i,:]))

-0.00010804415763365771
-0.00012076495158674927
-0.00016855579160998668
-0.0001749720061248461
-0.00015567783701361804
-0.0001281622263192006
-0.0001791103529397431
-8.910010416604552e-05
-0.00023081461784922597
-0.0002105346221784321


In [221]:
np.asarray(scores_spyeeg).shape

(20, 6)

In [222]:
np.asarray(scores_iRRR).shape

(11, 20)